# Week 12: No-Distillation Baseline (ViT from scratch)

Trains **the same ViT student** used in the DKD / feature-distillation notebooks on CIFAR-10 with **cross-entropy only** (no teacher). Same data pipeline, same augmentations, same LR schedule, same target accuracy and validation cadence, so the **time-to-85%** here is a clean apples-to-apples baseline to compare against vanilla KD, DKD, and DKD + feature distillation.

This is **not** a maximally-tuned run — it is a safe, stable config that reliably trains. The point is: *how much does the distillation signal help?*

## Imports

In [ ]:
import sys
import math
import time
from pathlib import Path
from typing import Dict, List, Optional

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data"

import torch
import torch.nn as nn
import torch.nn.functional as F

from src.model import count_parameters
from src.utils import (
    get_device,
    get_cifar10_loaders,
    get_model,
    set_seed,
    validate,
    get_peak_gpu_memory_mb,
    reset_peak_gpu_memory,
)

torch.set_float32_matmul_precision("high")
print(torch.__version__, get_device())

## Config

Matches the DKD notebook exactly except: no teacher, no KD terms.

In [ ]:
SEED = 42
TARGET_ACC = 85.0
MAX_EPOCHS = 100

BATCH_SIZE = 512
LR_REF_BATCH = 512
LR_REF = 1e-3
LR = LR_REF * (BATCH_SIZE / LR_REF_BATCH)
NUM_WORKERS = 4
WARMUP_EPOCHS = 5

LABEL_SMOOTHING = 0.1  # mild; matches common ViT recipes

VALIDATE_EVERY_COARSE = 2
VALIDATE_DENSE_AFTER = 74.0

set_seed(SEED, deterministic=False)
print(f"batch={BATCH_SIZE} LR={LR:.2e} CE-only label_smoothing={LABEL_SMOOTHING}")

## Data

In [ ]:
train_loader, test_loader = get_cifar10_loaders(
    data_dir=str(DATA_DIR),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    augment_train=True,
    use_randaugment=True,
    randaugment_num_ops=2,
    randaugment_magnitude=9,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2,
)
xb, _ = next(iter(train_loader))
device = get_device()
print(len(train_loader), "batches", xb.shape)

## Student

In [ ]:
def make_student():
    return get_model(patch_size=4, embed_dim=192, depth=6, num_heads=6, dropout=0.0)

print("student params", count_parameters(make_student()))

## Train: cross-entropy only

In [ ]:
def train_baseline(student, train_loader, test_loader):
    device = get_device()
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
    student = student.to(device)

    opt = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.01)

    def lr_lambda(ep):
        if ep < WARMUP_EPOCHS:
            return (ep + 1) / WARMUP_EPOCHS
        p = (ep - WARMUP_EPOCHS) / max(1, MAX_EPOCHS - WARMUP_EPOCHS)
        return 0.5 * (1 + math.cos(math.pi * p))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    use_amp = device.type == "cuda"
    dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16
    ce_loss = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    hist: Dict[str, List] = {"test_acc": [], "train_loss": [], "wall_time": []}
    best, ttt = 0.0, None
    last_val: Optional[float] = None
    t0 = time.perf_counter()

    for ep in range(1, MAX_EPOCHS + 1):
        student.train()
        tot = 0.0
        t_ep = time.perf_counter()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            if use_amp:
                with torch.amp.autocast(device_type="cuda", dtype=dtype):
                    zs = student(x)
            else:
                zs = student(x)
            loss = ce_loss(zs.float(), y)
            loss.backward()
            nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            opt.step()
            tot += loss.item()
        sched.step()
        wt = time.perf_counter() - t_ep

        run_val = (ep == 1) or (ep == MAX_EPOCHS)
        if last_val is not None and last_val >= VALIDATE_DENSE_AFTER:
            run_val = True
        elif ep % VALIDATE_EVERY_COARSE == 0:
            run_val = True
        if run_val:
            _, acc = validate(student, test_loader, nn.CrossEntropyLoss(), device)
            last_val = acc
        else:
            acc = last_val if last_val is not None else 0.0

        best = max(best, acc)
        if ttt is None and acc >= TARGET_ACC:
            ttt = time.perf_counter() - t0
        hist["train_loss"].append(tot / len(train_loader))
        hist["test_acc"].append(acc)
        hist["wall_time"].append(wt)
        tag = "~" if not run_val else ""
        print(f"ep{ep:3d} loss {hist['train_loss'][-1]:.4f} test{tag} {acc:.2f}% ({wt:.1f}s/ep)")
        if ttt is not None:
            print(f"  >>> {TARGET_ACC}% in {ttt:.2f}s wall-clock")
            break
    return {"history": hist, "best_acc": best, "time_to_target": ttt, "total_time": time.perf_counter() - t0}


set_seed(SEED)
reset_peak_gpu_memory()
results = train_baseline(make_student(), train_loader, test_loader)
print(results["time_to_target"], results["best_acc"], "peak", get_peak_gpu_memory_mb())

## Plot

In [ ]:
import matplotlib.pyplot as plt

h = results["history"]
cum = [sum(h["wall_time"][: i + 1]) for i in range(len(h["wall_time"]))]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h["test_acc"], lw=2)
ax[0].axhline(TARGET_ACC, color="gray", ls="--")
ax[0].set_xlabel("epoch")
ax[0].set_ylabel("test %")
ax[0].set_title("Baseline (CE only): test accuracy")
ax[0].grid(alpha=0.3)
ax[1].plot(cum, h["test_acc"], lw=2)
ax[1].axhline(TARGET_ACC, color="gray", ls="--")
if results["time_to_target"]:
    ax[1].axvline(results["time_to_target"], color="r", ls=":", label="hit")
    ax[1].legend()
ax[1].set_xlabel("wall s")
ax[1].set_title("Time-to-target")
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "notebooks" / "no_distillation_baseline_results.png", dpi=150, bbox_inches="tight")
plt.show()